# Uni-09 — MATLAB engineering experiment

Read the corresponding note in `notes/05 Knowledge Syntheses`. Run cells in order. MATLAB and the required toolboxes must be installed and licensed locally.

## Select duration, scenario and scheduling policies

Choose 0.1 or 0.2 seconds and a positive step that divides this duration exactly. Scenario 1 is light load, 2 heavy load, 3 greater distance, and 4 more uplink slots. Policy 1 is RoundRobin, 2 ProportionalFair, and 3 BestCQI. The two policy runs share the original seed convention and scenario. This cell requires both 5G Toolbox and Wireless Network Toolbox.

In [ ]:
courseRoot=pwd;
while ~isfile(fullfile(courseRoot,'publish_site.py'))
    parent=fileparts(courseRoot);
    assert(~strcmp(parent,courseRoot),'Uni09:Root','Open the notebook inside the course package.');
    courseRoot=parent;
end
addpath(fullfile(courseRoot,'matlab'));
addpath(fullfile(courseRoot,'matlab','uni09','jupyter'));
assert(~isempty(which('wirelessNetworkSimulator')) && ~isempty(which('nrGNB')),'Uni09:Toolbox','Install and license 5G Toolbox and Wireless Network Toolbox.');
u09=struct;
u09.cfg=struct('seed',9007,'duration',0.1,'step',0.01,'frequency',3.5e9,'bandwidth',5e6, ...
    'scs',30e3,'txPower',10,'UEPower',23);
u09.scenario=2; u09.policyA=1; u09.policyB=2;
disp(version); disp(ver); disp(u09.cfg);

## Inspect positions before running the cell

Edit the four [east north height] rows below to change the simulated UE positions. Scenario 3 multiplies their horizontal coordinates by four. The plot and both simulations use the resulting u09.cfg.positions array. The gNB height stays at 10 m. The simulator uses its default free-space path loss with abstract PHY.

In [ ]:
assert(exist('u09','var')==1 && isfield(u09,'cfg'),'Uni09:Order','Run setup first.');
u09.cfg.positions=[30 0 1.5;100 100 1.5;300 -100 1.5;700 0 1.5];
if u09.scenario==3, u09.cfg.positions(:,1:2)=u09.cfg.positions(:,1:2)*4; end
disp(table((1:4)',vecnorm(u09.cfg.positions-[0 0 10],2,2),'VariableNames',{'UE','DistanceM'}));
figure('Theme','light','Color','w','Position',[100 100 850 460]);
scatter(u09.cfg.positions(:,1),u09.cfg.positions(:,2),80,'filled'); hold on; scatter(0,0,110,'^','filled');
text(u09.cfg.positions(:,1)+10,u09.cfg.positions(:,2),string(1:4)); hold off;
axis equal; grid on; xlabel('East (m)'); ylabel('North (m)'); legend('UE','gNB'); title('Connected cell geometry');
set(gca,'FontName','Helvetica','FontSize',14);

## Run the first scheduling policy

The worker creates a fresh simulator, directly connects four UEs, adds application traffic and advances in 10 ms steps. It records actual application events and protocol statistics. This cell executes the complete short run once; subsequent inspection uses its retained arrays.

In [ ]:
assert(exist('u09','var')==1 && isfield(u09.cfg,'positions'),'Uni09:Order','Inspect the geometry first.');
u09.first=uni09_network_experiment(u09.cfg,u09.scenario,u09.policyA);
disp(u09.first.pattern); disp(u09.first.results);
fprintf('%s / %s: %d application events\n',u09.first.scenarioName,u09.first.policyName,size(u09.first.events{1},1));

## Follow measured payload delivery through time

The cumulative curves sum the bytes received by the application at all four UEs and at the gNB. Goodput divides the final delivered bytes by this configured simulation duration. Generated bytes can remain queued or unreceived when the short run ends.

In [ ]:
assert(exist('u09','var')==1 && isfield(u09,'first'),'Uni09:Order','Run the first policy first.');
[u09.times,~,u09.timeIndex]=unique(u09.first.timeline.TimeS);
u09.dlBytes=accumarray(u09.timeIndex,u09.first.timeline.DLDeliveredBytes);
u09.ulBytes=accumarray(u09.timeIndex,u09.first.timeline.ULDeliveredBytes);
figure('Theme','light','Color','w','Position',[100 100 850 460]);
plot(u09.times*1e3,[u09.dlBytes u09.ulBytes]/1000,'-o','LineWidth',1.5);
xlabel('Elapsed simulation time (ms)'); ylabel('Delivered application bytes (kB)');
legend('Downlink','Uplink'); grid on; title(u09.first.policyName+' cumulative delivery');
set(gca,'FontName','Helvetica','FontSize',14);

## Run the second policy and compare per UE goodput

The second run rebuilds the cell with the same scenario and random seed convention. Its scheduler uses the policy selected in setup. Compare both link directions per UE before judging the aggregate result. Changing the scenario or either policy requires rerunning from setup.

In [ ]:
assert(exist('u09','var')==1 && isfield(u09,'first'),'Uni09:Order','Run the first policy first.');
u09.second=uni09_network_experiment(u09.cfg,u09.scenario,u09.policyB);
figure('Theme','light','Color','w','Position',[100 100 850 460]);
tiledlayout(1,2);
nexttile; bar([u09.first.results.DLGoodputMbps u09.second.results.DLGoodputMbps]);
xlabel('UE'); ylabel('Downlink goodput (Mbit/s)'); title('Downlink'); grid on;
legend([u09.first.policyName u09.second.policyName]); set(gca,'FontName','Helvetica','FontSize',13);
nexttile; bar([u09.first.results.ULGoodputMbps u09.second.results.ULGoodputMbps]);
xlabel('UE'); ylabel('Uplink goodput (Mbit/s)'); title('Uplink'); grid on;
legend([u09.first.policyName u09.second.policyName]); set(gca,'FontName','Helvetica','FontSize',13);

## Check fairness, retransmissions and undefined delay

Jain fairness compares all four UE goodputs. A direction with no delivered bytes has undefined fairness; an individual UE with no delivery has undefined mean delivered packet delay. The retransmission counters measure MAC activity. These two short runs do not provide confidence intervals or an access success probability.

In [ ]:
assert(exist('u09','var')==1 && isfield(u09,'second'),'Uni09:Order','Run the policy comparison first.');
u09.summary=zeros(2,6);
u09.checks=true;
for k=1:2
    if k==1, q=u09.first.results; else, q=u09.second.results; end
    assert(all(q.DLDeliveredBytes<=q.DLGeneratedBytes) && all(q.ULDeliveredBytes<=q.ULGeneratedBytes),'Uni09:Bytes','Delivered bytes exceed generated bytes.');
    assert(all(isnan(q.DLDelayMs(q.DLDeliveredBytes==0))) && all(isnan(q.ULDelayMs(q.ULDeliveredBytes==0))),'Uni09:Delay','No delivery must have undefined delay.');
    assert(all(isnan(q.DLJain) | (q.DLJain>=0.25-1e-12 & q.DLJain<=1+1e-12)),'Uni09:Fairness','Check the downlink fairness calculation.');
    assert(all(isnan(q.ULJain) | (q.ULJain>=0.25-1e-12 & q.ULJain<=1+1e-12)),'Uni09:Fairness','Check the uplink fairness calculation.');
    u09.summary(k,:)=[sum(q.DLGoodputMbps) sum(q.ULGoodputMbps) q.DLJain(1) q.ULJain(1) sum(q.DLRetransmissions) sum(q.ULRetransmissions)];
    disp(q(:,{'UE','DLDelayMs','ULDelayMs','DLDeliveredBytes','ULDeliveredBytes'}));
end
disp(array2table(u09.summary,'VariableNames',{'DLGoodputMbps','ULGoodputMbps','DLJain','ULJain','DLRetransmissions','ULRetransmissions'}, ...
    'RowNames',{char(u09.first.policyName+' first'),char(u09.second.policyName+' second')}));
figure('Theme','light','Color','w','Position',[100 100 850 460]);
bar(u09.summary(:,5:6)); xticks(1:2); xticklabels([u09.first.policyName u09.second.policyName]);
ylabel('MAC retransmission count'); legend('Downlink','Uplink'); title('Actual protocol counters during the short runs'); grid on;
set(gca,'FontName','Helvetica','FontSize',14);

## Save both simulations and their software information

Save the configurations, event logs, timeline and per UE results to a new student directory. Download the notebook to retain edited cells and figures. The saved arrays retain the abstract PHY and direct connection boundary of these experiments.

In [ ]:
assert(exist('u09','var')==1 && isfield(u09,'checks') && u09.checks,'Uni09:Order','Assess both simulations before saving.');
studentRoot=fullfile(courseRoot,'matlab','uni09','jupyter','student-results');
if ~isfolder(studentRoot), mkdir(studentRoot); end
u09.runDir=tempname(studentRoot); mkdir(u09.runDir);
software=ver;
save(fullfile(u09.runDir,'run.mat'),'u09','software');
writetable(u09.first.results,fullfile(u09.runDir,'first-policy.csv'));
writetable(u09.second.results,fullfile(u09.runDir,'second-policy.csv'));
fprintf('Student results saved to %s\n',u09.runDir);